%% [markdown]
# 03b — NHS ODS Hospital Locations Audit

In [1]:
# %%
import os, pandas as pd, numpy as np, re
from pathlib import Path
from pyproj import Transformer

In [2]:
# Run from project root
os.chdir(Path(__file__).parent.parent if '__file__' in dir() else Path.cwd())
ROOT = Path("/Users/souravamseekarmarti/Projects/aequitas")
DATA = ROOT / "data"

In [3]:
# %%
hospitals = pd.read_csv(DATA / "raw/poi/hospitals.csv", skiprows=14)
print(f"Rows: {len(hospitals)}, Cols: {hospitals.columns.tolist()}")
print(hospitals.head(2))

Rows: 3870, Cols: ['OrgId', 'Name', 'PostCode', 'LastChangeDate']
   OrgId                        Name PostCode LastChangeDate
0  A0E3V  ROYAL ORTHOPAEDIC HOSPITAL  B31 2AP     2024-05-08
1  A0F9W               SKIN HOSPITAL  B71 4HJ     2024-05-08


In [4]:
# %%
assert len(hospitals) == 3870, f"FAIL: expected 3870, got {len(hospitals)}"
print(f"CHECK PASS: {len(hospitals)} rows")
assert hospitals['OrgId'].nunique() == len(hospitals), "FAIL: duplicate OrgIds"
null_check = hospitals[['OrgId', 'Name', 'PostCode']].isnull().sum()
assert null_check.sum() == 0, f"FAIL: nulls: {null_check[null_check>0]}"
print("CHECK PASS: no nulls, no duplicates")

CHECK PASS: 3870 rows
CHECK PASS: no nulls, no duplicates


In [5]:
# %%
def standardise_postcode(pc):
    if pd.isna(pc): return None
    pc = re.sub(r'\s+', '', str(pc).strip().upper())
    return pc[:-3] + ' ' + pc[-3:] if len(pc) > 3 else pc

In [6]:
hospitals['postcode_std'] = hospitals['PostCode'].apply(standardise_postcode)
print(f"Sample: {hospitals['postcode_std'].head(3).tolist()}")

Sample: ['B31 2AP', 'B71 4HJ', 'BA11 2FH']


In [7]:
# %%
lookup = pd.read_parquet(DATA / "audit/postcode_lookup.parquet")
print(f"Lookup: {len(lookup):,} postcodes")

Lookup: 1,492,016 postcodes


In [8]:
geocoded = hospitals.merge(lookup.rename(columns={'Postcode': 'postcode_std'}), on='postcode_std', how='left')
matched = geocoded['Easting'].notna().sum()
match_rate = matched / len(geocoded) * 100
print(f"Match rate: {match_rate:.1f}% ({matched}/{len(geocoded)})")
assert match_rate >= 95.0, f"FAIL: match rate {match_rate:.1f}% below 95%"
print("CHECK PASS: match rate >= 95%")

Match rate: 96.0% (3714/3870)
CHECK PASS: match rate >= 95%


In [9]:
# %%
t = Transformer.from_crs("EPSG:27700", "EPSG:4326", always_xy=True)
has_coords = geocoded['Easting'].notna()
lons, lats = t.transform(geocoded.loc[has_coords, 'Easting'].values, geocoded.loc[has_coords, 'Northing'].values)
geocoded.loc[has_coords, 'longitude'] = lons
geocoded.loc[has_coords, 'latitude'] = lats
print(f"Lat: {geocoded['latitude'].min():.3f} to {geocoded['latitude'].max():.3f}")
print(f"Lon: {geocoded['longitude'].min():.3f} to {geocoded['longitude'].max():.3f}")
assert geocoded['latitude'].dropna().between(49.8, 55.8).all(), "FAIL: lat out of bounds"
assert geocoded['longitude'].dropna().between(-6.4, 1.8).all(), "FAIL: lon out of bounds"
print("CHECK PASS: all coords within England bounds")

Lat: 49.913 to 55.411
Lon: -6.309 to 1.730
CHECK PASS: all coords within England bounds


In [10]:
# %%
out = geocoded[['OrgId','Name','PostCode','postcode_std','Easting','Northing','latitude','longitude']].copy()
out.to_parquet(DATA / "audit/hospitals_geocoded.parquet", index=False)
print(f"Saved: {len(out)} rows, {out['latitude'].notna().sum()} geocoded ({match_rate:.1f}%)")
print(f"Northernmost: {out.loc[out['latitude'].idxmax(), 'Name']} ({out['latitude'].max():.4f}N)")
print(f"Southernmost: {out.loc[out['latitude'].idxmin(), 'Name']} ({out['latitude'].min():.4f}N)")
print("03b_hospitals COMPLETE")

Saved: 3870 rows, 3714 geocoded (96.0%)
Northernmost: ALNWICK INFIRMARY ELECTIVE SURGICAL HUB (55.4109N)
Southernmost: ST MARY'S HOSPITAL (49.9131N)
03b_hospitals COMPLETE
